# Subtitle SRT to Text Cleaner

This notebook reads SRT subtitle files from the `Sub/` folder, cleans them by removing timestamps, metadata, formatting tags, non-speech parentheticals, and speaker labels, and writes the clean dialogue lines to `.txt` files in the `Cleaned_Text/` directory.

In [1]:
import os
import re
import glob

# Define input and output paths
SUB_DIR = "Sub"
OUTPUT_DIR = "Cleaned_Text"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Directories configured: Input={SUB_DIR}, Output={OUTPUT_DIR}")

Directories configured: Input=Sub, Output=Cleaned_Text


In [2]:
def clean_subtitle_line(line):
    # 1. Remove HTML tags like <i>, </i>, <b>, </b>, etc.
    line = re.sub(r'<[^>]+>', '', line)
    
    # 2. Remove parentheticals representing sound effects or stage directions: (...) or [...]
    line = re.sub(r'\([^)]*\)', '', line)
    line = re.sub(r'\[[^\]]*\]', '', line)
    
    # 3. Remove music symbols and other noise characters
    line = re.sub(r'[♪#]', '', line)
    
    # 4. Remove speaker labels (e.g., "MARY:", "ADULT SHELDON:", "- MARY:")
    # Matches optional leading dash/spaces, followed by uppercase word(s) and a colon.
    line = re.sub(r'^\s*(?:-\s*)?[A-Z][A-Z0-9\s\'\.\-]{0,25}:\s*', '', line)
    
    # 5. Remove leading dialogue dashes (e.g., "- Coming!")
    line = re.sub(r'^\s*-\s*', '', line)
    
    # 6. Normalize whitespace
    line = re.sub(r'\s+', ' ', line).strip()
    
    # 7. Clean any remaining leading colons and whitespace (e.g. from removing '(sighs):')
    line = re.sub(r'^[:\s]+', '', line).strip()
    
    return line

def parse_srt_file(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    # Split blocks by double-newlines
    blocks = re.split(r'\n\s*\n', content)
    cleaned_lines = []
    
    for block in blocks:
        lines = [line.strip() for line in block.split('\n') if line.strip()]
        if not lines:
            continue
            
        # Check if the block has a timestamp (e.g., "00:00:04,004 --> 00:00:06,166")
        time_idx = -1
        for idx, line in enumerate(lines):
            if '-->' in line:
                time_idx = idx
                break
                
        if time_idx != -1:
            # Everything after the timestamp line is text
            text_lines = lines[time_idx + 1:]
        else:
            # Fallback if no timestamp line is found (filter out numeric lines)
            text_lines = [line for line in lines if not line.isdigit()]
        
        # Join multi-line subtitles in the block into a single line
        block_text = " ".join(text_lines)
        cleaned = clean_subtitle_line(block_text)
        if cleaned:
            cleaned_lines.append(cleaned)
            
    return cleaned_lines

In [ ]:
srt_files = glob.glob(os.path.join(SUB_DIR, "*.srt"))
print(f"Found {len(srt_files)} subtitle (.srt) files in '{SUB_DIR}'.")

for filepath in sorted(srt_files):
    filename = os.path.basename(filepath)
    # Extract episode name (e.g., S01E01)
    match = re.search(r'S\d+E\d+', filename, re.IGNORECASE)
    if match:
        ep_name = match.group(0).upper()
    else:
        ep_name = os.path.splitext(filename)[0]
        
    cleaned_lines = parse_srt_file(filepath)
    
    output_path = os.path.join(OUTPUT_DIR, f"{ep_name}.txt")
    with open(output_path, 'w', encoding='utf-8') as f:
        for line in cleaned_lines:
            f.write(line + "\n")
            
    print(f"Processed: {filename} -> {ep_name}.txt ({len(cleaned_lines)} lines)")
print("\nAll subtitle files successfully cleaned and exported to 'Cleaned_Text/'!")